In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "india"
vehicle = "rice"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,baseline,0,141,13.687242
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,baseline,0,141,4.700669
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,baseline,0,141,8.710063
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,baseline,0,141,10.645632
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,baseline,0,141,8.387468
...,...,...,...,...,...,...,...,...,...,...,...
1079995,person_time,impairment,anemia,severe,95_plus,severe,1,zero,0,129,0.000000
1079996,person_time,impairment,anemia,severe,95_plus,severe,2,zero,0,129,0.000000
1079997,person_time,impairment,anemia,severe,95_plus,severe,3,zero,0,129,0.000000
1079998,person_time,impairment,anemia,severe,95_plus,severe,4,zero,0,129,0.000000


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

not_anemic    270000
mild          270000
moderate      270000
severe        270000
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  4.931432e+06
              2                  4.078843e+06
              3                  3.655300e+06
              4                  3.475433e+06
              5                  3.393418e+06
intervention  1                  4.931446e+06
              2                  4.078846e+06
              3                  3.655309e+06
              4                  3.475438e+06
              5                  3.393421e+06
zero          1                  4.931403e+06
              2                  4.078826e+06
              3                  3.655286e+06
              4                  3.475416e+06
              5                  3.393414e+06
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  2.635997e+06
              2                  2.080789e+06
              3                  1.777292e+06
              4                  1.566441e+06
              5                  1.320718e+06
intervention  1                  2.552408e+06
              2                  2.013032e+06
              3                  1.714640e+06
              4                  1.508149e+06
              5                  1.279352e+06
zero          1                  2.825098e+06
              2                  2.220594e+06
              3                  1.891403e+06
              4                  1.664846e+06
              5                  1.377210e+06
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.534530
              2                  0.510142
              3                  0.486223
              4                  0.450718
              5                  0.389200
intervention  1                  0.517578
              2                  0.493530
              3                  0.469082
              4                  0.433945
              5                  0.377009
zero          1                  0.572879
              2                  0.544420
              3                  0.517443
              4                  0.479035
              5                  0.405848
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,48678.711021
1,Female,0.0,0.019178,not_pregnant,2,42723.422230
2,Female,0.0,0.019178,not_pregnant,3,37932.716572
3,Female,0.0,0.019178,not_pregnant,4,35728.167677
4,Female,0.0,0.019178,not_pregnant,5,28631.476092
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,18328.880607
281,Male,95.0,125.000000,not_pregnant,2,19140.234171
282,Male,95.0,125.000000,not_pregnant,3,19770.250303
283,Male,95.0,125.000000,not_pregnant,4,20851.307187


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    4.478246e+06
2    3.707329e+06
3    3.317141e+06
4    3.157155e+06
5    3.080667e+06
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.393756e+06
              2                  1.891264e+06
              3                  1.612872e+06
              4                  1.422988e+06
              5                  1.198996e+06
intervention  1                  2.317842e+06
              2                  1.829677e+06
              3                  1.556012e+06
              4                  1.370031e+06
              5                  1.161441e+06
zero          1                  2.565494e+06
              2                  2.018343e+06
              3                  1.716432e+06
              4                  1.512387e+06
              5                  1.250283e+06
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,baseline,0,141,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,baseline,0,141,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,baseline,0,141,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,baseline,0,141,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,zero,0,129,0.0
539996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,zero,0,129,0.0
539997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,zero,0,129,0.0
539998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,zero,0,129,0.0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['susceptible_to_maternal_disorders_to_maternal_disorders', 'maternal_disorders_to_recovered_from_maternal_disorders'], dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.933208e+06
              2                  1.741533e+06
              3                  2.212491e+06
              4                  1.981041e+06
              5                  1.262527e+06
intervention  1                  2.896727e+06
              2                  1.719550e+06
              3                  2.186499e+06
              4                  1.958401e+06
              5                  1.251882e+06
zero          1                  3.016543e+06
              2                  1.788523e+06
              3                  2.261373e+06
              4                  2.020071e+06
              5                  1.277061e+06
Name: value, dtype: float64

In [18]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_scenario below, so we'd need to
# change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if pathlib.Path(path).is_file():
    neonatal_deaths = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,deaths,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,baseline,0,81,428.812375
1,deaths,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,baseline,0,81,346.904393
2,deaths,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,baseline,0,81,289.086994
3,deaths,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,baseline,0,81,228.860537
4,deaths,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,baseline,0,81,233.678654
...,...,...,...,...,...,...,...,...,...,...,...,...
23995,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,71,40.953991
23996,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,71,24.090583
23997,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,71,40.953991
23998,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,71,31.317758


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  183844.874040
              2                  151867.034351
              3                  134883.173435
              4                  128593.122250
              5                  121891.122098
intervention  1                  183413.652607
              2                  151536.993366
              3                  134675.994422
              4                  128371.488888
              5                  121775.487301
zero          1                  184370.048747
              2                  152242.847444
              3                  135116.852089
              4                  128855.709603
              5                  121958.575730
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,39601.769161,zero
1,Female,0.0,0.019178,2,32639.498130,zero
2,Female,0.0,0.019178,3,28574.791970,zero
3,Female,0.0,0.019178,4,25160.080603,zero
4,Female,0.0,0.019178,5,18957.590376,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,8038.471642,intervention
746,Male,95.0,125.000000,2,7603.438965,intervention
747,Male,95.0,125.000000,3,7671.327456,intervention
748,Male,95.0,125.000000,4,7551.154222,intervention


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.224695e+08
              2                  1.150335e+08
              3                  1.138088e+08
              4                  1.086529e+08
              5                  1.031774e+08
intervention  1                  1.168274e+08
              2                  1.096889e+08
              3                  1.087321e+08
              4                  1.038294e+08
              5                  9.984783e+07
zero          1                  1.291309e+08
              2                  1.213341e+08
              3                  1.193293e+08
              4                  1.134219e+08
              5                  1.058060e+08
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.248633e+08
              2                  1.169248e+08
              3                  1.154217e+08
              4                  1.100759e+08
              5                  1.043764e+08
intervention  1                  1.191453e+08
              2                  1.115186e+08
              3                  1.102881e+08
              4                  1.051994e+08
              5                  1.010093e+08
zero          1                  1.316964e+08
              2                  1.233524e+08
              3                  1.210458e+08
              4                  1.149343e+08
              5                  1.070563e+08
Name: value, dtype: float64

In [25]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  4684.597214
              2                  4138.352740
              3                  3715.268143
              4                  3503.314763
              5                  2983.626124
baseline      1                  4354.336021
              2                  3889.504137
              3                  3526.334979
              4                  3348.459399
              5                  2924.573729
intervention  1                  1635.964576
              2                  1650.239767
              3                  1627.774492
              4                  1635.480006
              5                  1964.984223
Name: value, dtype: float64

In [27]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)